# 01 · Pipeline ETL SmartPort : SOURCES → RAW → STAGING → CORE → MART

**Objectif.** Transformer les 5 fichiers CSV du terminal en un entrepôt de données fiable, contrôlé et traçable.

| § | Étape | Résultat |
|---|---|---|
| 0 | Préparation | chemins, journal, fonctions |
| 1 | RAW | copie fidèle des CSV |
| 2 | STAGING | nettoyage, typage, rejets |
| 3 | CORE | schéma en étoile : 2 faits, 8 dimensions |
| 4 | MART | 7 tables d'indicateurs |
| 5 | Qualité | 32 contrôles et réconciliation |
| 6 | Monitoring | volumes et durées |
| 7 | Idempotence | deux exécutions, même résultat |
| 8 | Test des rejets | 13 erreurs injectées |
| 9 | Lineage | de la colonne CSV à l'indicateur |
| 10 | Conclusion | bilan et limites |

**Exécution :** *Run All* (le notebook retrouve seul la racine du projet). La base produite est `03_data/warehouse/smartport_complet.db`, lue ensuite par `02_EDA_Analyse_Exploratoire.ipynb`.
L'exploration SQL des sources (18 requêtes) se trouve dans `04_scripts/sql/exploration/`.

## 0. Préparation
### 0.1 Bibliothèques et chemins
Le notebook retrouve la racine du projet, ce qui évite tout chemin propre à un ordinateur.
`STEPS` et `CHECKS` mémorisent les étapes et les contrôles pour pouvoir les **rejouer** (§ 7).
`REBUILT` liste les tables reconstruites à chaque exécution.

In [1]:
# --- Racine du projet : on remonte jusqu'au dossier qui contient 03_data/raw
from pathlib import Path
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "03_data" / "raw").is_dir()), None)
assert ROOT is not None, "Lancez le notebook depuis le dossier du projet (celui qui contient 03_data/)."
DATA_DIR = ROOT / "03_data" / "raw"                                  # CSV sources (lecture seule)
DB_PATH  = ROOT / "03_data" / "warehouse" / "smartport_complet.db"   # entrepôt produit

import re, os, math, shutil, sqlite3, time
from datetime import datetime
import pandas as pd

pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", 90)
pd.set_option("display.max_rows", 60)

SOURCES = ["customers", "vessel_calls", "container_events", "yard_moves", "equipment"]

# Tables reconstruites à chaque exécution (le journal etl_run_metrics, lui, est conservé)
REBUILT = [
    "raw_customers", "raw_vessel_calls", "raw_container_events", "raw_yard_moves",
    "raw_equipment", "stg_customers", "stg_vessels", "stg_equipment",
    "stg_container_events", "stg_yard_moves", "fact_yard_move", "fact_container_event",
    "dim_equipment", "dim_freight_status", "dim_event_type", "dim_yard_area",
    "dim_vessel", "dim_customer", "dim_hour", "dim_date",
    "mart_kpi_events_per_day", "mart_kpi_move_by_area", "mart_kpi_vessel", "mart_kpi_service",
    "mart_kpi_customer", "mart_kpi_event_type", "mart_kpi_freight", "etl_rejects",
    "etl_quality_checks",
]

STEPS = []    # mémoire des étapes SQL exécutées
CHECKS = []   # mémoire des contrôles qualité exécutés

### 0.2 Connexion et journal d'exécution
- `connect` ouvre la base SQLite et crée le journal `etl_run_metrics` : une ligne par étape.
- `reset_layers` supprime les couches avant reconstruction. Le journal est conservé.

In [2]:
# Le journal n'est jamais supprimé : il garde l'historique de toutes les exécutions
def connect(db_path="smartport_etl.db"):
    con = sqlite3.connect(db_path)
    con.execute("""CREATE TABLE IF NOT EXISTS etl_run_metrics (
        run_id TEXT, layer TEXT, step TEXT, rows_affected INTEGER,
        duration_s REAL, status TEXT, logged_at TEXT)""")
    con.commit()
    return con

def reset_layers(con):
    for t in REBUILT:
        con.execute(f"DROP TABLE IF EXISTS {t}")
    con.commit()

def new_run_id():
    return datetime.now().strftime("%Y%m%d-%H%M%S")

def log_metric(con, run_id, layer, step, rows, duration, status="OK"):
    con.execute("INSERT INTO etl_run_metrics VALUES (?,?,?,?,?,?,?)",
                (run_id, layer, step, rows, round(duration, 4), status,
                 datetime.now().isoformat(timespec="seconds")))
    con.commit()

### 0.3 Exécuter et mesurer une étape
`run_step` exécute une requête, mesure sa **durée** et le **nombre de lignes** produites, puis les écrit dans le journal.

In [3]:
# Une seule porte d'entrée pour le SQL du pipeline : chaque étape est donc journalisée
def run_step(con, run_id, layer, step, sql, register=True, verbose=True):
    t0 = time.perf_counter()
    before = con.total_changes
    con.execute(sql)
    con.commit()
    m = re.match(r"\s*CREATE TABLE (\w+) AS", sql, re.I)
    if m:                                    # CREATE TABLE ... AS SELECT : on compte les lignes créées
        rows = con.execute(f"SELECT COUNT(*) FROM {m.group(1)}").fetchone()[0]
    elif re.match(r"\s*(WITH|INSERT)", sql, re.I):
        rows = con.total_changes - before
    else:                                    # CREATE TABLE vide, DROP...
        rows = None
    dt = time.perf_counter() - t0
    log_metric(con, run_id, layer, step, rows, dt)
    if register and not any(s[1] == step for s in STEPS):
        STEPS.append((layer, step, sql))
    if verbose:
        print(f"[{layer:<7}] {step:<36} lignes={rows if rows is not None else '-':>6}   {dt*1000:7.1f} ms")
    return rows

def load_raw(con, run_id, data_dir=".", verbose=True):
    for s in SOURCES:
        t0 = time.perf_counter()
        df = pd.read_csv(f"{data_dir}/{s}.csv", dtype=str, keep_default_na=False)
        df["_source_file"] = f"{s}.csv"
        df["_loaded_at"] = datetime.now().isoformat(timespec="seconds")
        df.to_sql(f"raw_{s}", con, if_exists="replace", index=False)
        log_metric(con, run_id, "RAW", f"load raw_{s}", len(df), time.perf_counter() - t0)
        if verbose:
            print(f"[RAW    ] load raw_{s:<30} lignes={len(df):>6}")

### 0.4 Démarrage
On repart de couches vides et on crée `q()`, qui renvoie le résultat d'une requête sous forme de tableau.

In [4]:
# SQRT n'est pas toujours disponible dans SQLite : on la fournit depuis Python
con = connect(DB_PATH)
con.create_function("SQRT", 1, math.sqrt)
reset_layers(con)            # idempotence : aucune table d'une exécution précédente ne subsiste
run_id = new_run_id()        # identifiant de cette exécution dans le journal
print("run_id =", run_id)

def q(sql):
    return pd.read_sql(sql, con)

run_id = 20261125-214647


## 1. Couche RAW : copie fidèle
Chaque CSV devient une table `raw_*`, **sans aucune transformation** : tout reste en texte.
Deux colonnes de traçabilité sont ajoutées : `_source_file` et `_loaded_at`.

*Pourquoi ?* Conserver la donnée telle qu'elle est arrivée permet de rejouer les règles et d'expliquer un rejet.

In [5]:
# Lecture en texte (dtype=str) : aucune conversion implicite par pandas
load_raw(con, run_id, DATA_DIR)

[RAW    ] load raw_customers                      lignes=    80
[RAW    ] load raw_vessel_calls                   lignes=   120
[RAW    ] load raw_container_events               lignes=  9000
[RAW    ] load raw_yard_moves                     lignes=  4500
[RAW    ] load raw_equipment                      lignes=    40


In [6]:
q("SELECT * FROM raw_container_events LIMIT 5")

,event_id,container_id,customer_id,vessel_id,event_type,event_ts,yard_area,freight_status,gross_weight_kg,_source_file,_loaded_at
0,1,CONT0003316,C050,V030,MOVE,2026-03-25 01:47:32,Y08B,FCL,8822.5,container_events.csv,2026-11-25T21:46:47
1,2,CONT0000650,C079,V074,SEAL,2026-03-19 23:42:20,Y02,FCL,24343.6,container_events.csv,2026-11-25T21:46:47
2,3,CONT0000269,C013,V006,SEAL,2026-03-20 15:15:26,Y04,FCL,20569.2,container_events.csv,2026-11-25T21:46:47
3,4,CONT0001082,C051,V038,DELIVER,2026-06-18 09:43:51,Y08B,FCL,30817.3,container_events.csv,2026-11-25T21:46:47
4,5,CONT0002622,C027,V095,DISCHARGE,2026-06-26 09:49:04,Y08,MTY,25745.9,container_events.csv,2026-11-25T21:46:47


**Interprétation.** Les cinq sources sont chargées en entier : 80 clients, 120 escales, 9 000 événements, 4 500 mouvements et 40 équipements. La couche RAW est la référence de la réconciliation (§ 5).

## 2. Couche STAGING
On **nettoie et on contrôle** : suppression des espaces inutiles (`TRIM`), typage des dates et des nombres, standardisation de la zone (`Y08B` → `Y08`), puis **règles de rejet**.
Chaque ligne qui viole une règle est copiée dans `etl_rejects` avec l'identifiant de la règle, et n'entre pas dans le DWH.

Règles : `EV01` à `EV09` pour les événements, `MV01` à `MV06` pour les mouvements de cour (`MV04` et `MV06` correspondent aux règles `FK_EQUIPMENT` et `FK_CONTAINER` du prof).
Une anomalie n'est **pas** rejetée mais signalée plus loin (contrôle Q13) : les mouvements dont la zone de départ est identique à la zone d'arrivée. Les rejeter supprimerait 16 % des mouvements.

**Référentiel clients.** Espaces supprimés et une seule ligne par `customer_id` (la première).

In [7]:
# [STAGING] stg_customers : étape exécutée, chronométrée et journalisée
run_step(con, run_id, "STAGING", "stg_customers", """
CREATE TABLE stg_customers AS
SELECT TRIM(customer_id) AS customer_id, TRIM(customer_name) AS customer_name,
       TRIM(group_code) AS group_code, TRIM(customer_type) AS customer_type
FROM raw_customers
WHERE TRIM(customer_id) <> ''
  AND rowid IN (SELECT MIN(rowid) FROM raw_customers GROUP BY TRIM(customer_id))
""")

[STAGING] stg_customers                        lignes=    80       3.1 ms


80

**Référentiel navires.** Même traitement sur `vessel_id`.

In [8]:
# [STAGING] stg_vessels : étape exécutée, chronométrée et journalisée
run_step(con, run_id, "STAGING", "stg_vessels", """
CREATE TABLE stg_vessels AS
SELECT TRIM(vessel_id) AS vessel_id, TRIM(vessel_name) AS vessel_name,
       TRIM(vessel_type) AS vessel_type, TRIM(service_code) AS service_code
FROM raw_vessel_calls
WHERE TRIM(vessel_id) <> ''
  AND rowid IN (SELECT MIN(rowid) FROM raw_vessel_calls GROUP BY TRIM(vessel_id))
""")

[STAGING] stg_vessels                          lignes=   120       1.9 ms


120

**Référentiel équipements.** Même traitement sur `equipment_id`.

In [9]:
# [STAGING] stg_equipment : étape exécutée, chronométrée et journalisée
run_step(con, run_id, "STAGING", "stg_equipment", """
CREATE TABLE stg_equipment AS
SELECT TRIM(equipment_id) AS equipment_id, TRIM(equipment_type) AS equipment_type,
       TRIM(home_area) AS home_area
FROM raw_equipment
WHERE TRIM(equipment_id) <> ''
  AND rowid IN (SELECT MIN(rowid) FROM raw_equipment GROUP BY TRIM(equipment_id))
""")

[STAGING] stg_equipment                        lignes=    40       1.6 ms


40

**Table des rejets.** Chaque rejet garde la table d'origine, le numéro de ligne RAW et la règle : c'est la **traçabilité des lignes écartées**. Les référentiels sont construits avant, car les règles EV07, EV08 et MV04 les consultent.

In [10]:
# [STAGING] create etl_rejects : étape exécutée, chronométrée et journalisée
run_step(con, run_id, "STAGING", "create etl_rejects", """
CREATE TABLE etl_rejects (
    reject_id INTEGER PRIMARY KEY AUTOINCREMENT,
    source_table TEXT, raw_rowid INTEGER, business_key TEXT,
    rule_id TEXT, rule_description TEXT)
""")

[STAGING] create etl_rejects                   lignes=     -       2.0 ms


**Règle EV01 event_id manquant.** Sans identifiant, le grain « 1 ligne = 1 événement » n'est pas garanti.

In [11]:
# [STAGING] EV01 event_id manquant : étape exécutée, chronométrée et journalisée
run_step(con, run_id, "STAGING", "EV01 event_id manquant", """
INSERT INTO etl_rejects (source_table, raw_rowid, business_key, rule_id, rule_description)
SELECT 'container_events', rowid, event_id, 'EV01', 'event_id manquant'
FROM raw_container_events WHERE event_id IS NULL OR TRIM(event_id) = ''
""")

[STAGING] EV01 event_id manquant               lignes=     0       2.5 ms


0

**Règle EV02 event_id en double.** Un doublon gonflerait les volumes.

In [12]:
# [STAGING] EV02 event_id en double : étape exécutée, chronométrée et journalisée
run_step(con, run_id, "STAGING", "EV02 event_id en double", """
INSERT INTO etl_rejects (source_table, raw_rowid, business_key, rule_id, rule_description)
SELECT 'container_events', rowid, event_id, 'EV02', 'event_id en double (on garde la 1re ligne)'
FROM raw_container_events WHERE TRIM(event_id) <> '' AND rowid NOT IN (SELECT MIN(rowid) FROM raw_container_events GROUP BY event_id)
""")

[STAGING] EV02 event_id en double              lignes=     0       7.6 ms


0

**Règle EV03 date invalide.** Sans date, l'événement ne peut pas être placé dans le temps.

In [13]:
# [STAGING] EV03 date invalide : étape exécutée, chronométrée et journalisée
run_step(con, run_id, "STAGING", "EV03 date invalide", """
INSERT INTO etl_rejects (source_table, raw_rowid, business_key, rule_id, rule_description)
SELECT 'container_events', rowid, event_id, 'EV03', 'event_ts illisible'
FROM raw_container_events WHERE datetime(event_ts) IS NULL
""")

[STAGING] EV03 date invalide                   lignes=     0       1.5 ms


0

**Règle EV04 type hors domaine.** Un type inconnu ne peut être rattaché à aucune opération.

In [14]:
# [STAGING] EV04 type hors domaine : étape exécutée, chronométrée et journalisée
run_step(con, run_id, "STAGING", "EV04 type hors domaine", """
INSERT INTO etl_rejects (source_table, raw_rowid, business_key, rule_id, rule_description)
SELECT 'container_events', rowid, event_id, 'EV04', 'event_type hors domaine'
FROM raw_container_events WHERE event_type NOT IN ('DISCHARGE','LOAD','MOVE','DELIVER','RECEIVE','STUFF','SEAL')
""")

[STAGING] EV04 type hors domaine               lignes=     0       1.9 ms


0

**Règle EV05 statut FCL/MTY invalide.** La part FCL / MTY doit rester exacte.

In [15]:
# [STAGING] EV05 statut FCL/MTY invalide : étape exécutée, chronométrée et journalisée
run_step(con, run_id, "STAGING", "EV05 statut FCL/MTY invalide", """
INSERT INTO etl_rejects (source_table, raw_rowid, business_key, rule_id, rule_description)
SELECT 'container_events', rowid, event_id, 'EV05', 'freight_status hors FCL/MTY'
FROM raw_container_events WHERE freight_status NOT IN ('FCL','MTY')
""")

[STAGING] EV05 statut FCL/MTY invalide         lignes=     0       1.2 ms


0

**Règle EV06 poids hors plage.** Seuil de 40 000 kg : hypothèse du projet. Une valeur non numérique est convertie en 0, donc rejetée.

In [16]:
# [STAGING] EV06 poids hors plage : étape exécutée, chronométrée et journalisée
run_step(con, run_id, "STAGING", "EV06 poids hors plage", """
INSERT INTO etl_rejects (source_table, raw_rowid, business_key, rule_id, rule_description)
SELECT 'container_events', rowid, event_id, 'EV06', 'poids non numerique ou hors 0..40000 kg'
FROM raw_container_events WHERE CAST(gross_weight_kg AS REAL) <= 0 OR CAST(gross_weight_kg AS REAL) > 40000
""")

[STAGING] EV06 poids hors plage                lignes=     0       1.5 ms


0

**Règle EV07 client inconnu.** Le fait doit pointer vers un client connu.

In [17]:
# [STAGING] EV07 client inconnu : étape exécutée, chronométrée et journalisée
run_step(con, run_id, "STAGING", "EV07 client inconnu", """
INSERT INTO etl_rejects (source_table, raw_rowid, business_key, rule_id, rule_description)
SELECT 'container_events', rowid, event_id, 'EV07', 'customer_id absent du referentiel'
FROM raw_container_events WHERE customer_id NOT IN (SELECT customer_id FROM stg_customers)
""")

[STAGING] EV07 client inconnu                  lignes=     0       2.7 ms


0

**Règle EV08 navire inconnu.** Le fait doit pointer vers un navire connu.

In [18]:
# [STAGING] EV08 navire inconnu : étape exécutée, chronométrée et journalisée
run_step(con, run_id, "STAGING", "EV08 navire inconnu", """
INSERT INTO etl_rejects (source_table, raw_rowid, business_key, rule_id, rule_description)
SELECT 'container_events', rowid, event_id, 'EV08', 'vessel_id absent du referentiel'
FROM raw_container_events WHERE vessel_id NOT IN (SELECT vessel_id FROM stg_vessels)
""")

[STAGING] EV08 navire inconnu                  lignes=     0       2.6 ms


0

**Règle EV09 zone manquante.** La zone est l'axe de l'indicateur MOVE par zone.

In [19]:
# [STAGING] EV09 zone manquante : étape exécutée, chronométrée et journalisée
run_step(con, run_id, "STAGING", "EV09 zone manquante", """
INSERT INTO etl_rejects (source_table, raw_rowid, business_key, rule_id, rule_description)
SELECT 'container_events', rowid, event_id, 'EV09', 'yard_area manquante'
FROM raw_container_events WHERE yard_area IS NULL OR TRIM(yard_area) = ''
""")

[STAGING] EV09 zone manquante                  lignes=     0       1.5 ms


0

**Règle MV01 move_id manquant.** Grain « 1 ligne = 1 mouvement ».

In [20]:
# [STAGING] MV01 move_id manquant : étape exécutée, chronométrée et journalisée
run_step(con, run_id, "STAGING", "MV01 move_id manquant", """
INSERT INTO etl_rejects (source_table, raw_rowid, business_key, rule_id, rule_description)
SELECT 'yard_moves', rowid, move_id, 'MV01', 'move_id manquant'
FROM raw_yard_moves WHERE move_id IS NULL OR TRIM(move_id) = ''
""")

[STAGING] MV01 move_id manquant                lignes=     0       0.7 ms


0

**Règle MV02 move_id en double.** Un doublon gonflerait les mouvements.

In [21]:
# [STAGING] MV02 move_id en double : étape exécutée, chronométrée et journalisée
run_step(con, run_id, "STAGING", "MV02 move_id en double", """
INSERT INTO etl_rejects (source_table, raw_rowid, business_key, rule_id, rule_description)
SELECT 'yard_moves', rowid, move_id, 'MV02', 'move_id en double (on garde la 1re ligne)'
FROM raw_yard_moves WHERE TRIM(move_id) <> '' AND rowid NOT IN (SELECT MIN(rowid) FROM raw_yard_moves GROUP BY move_id)
""")

[STAGING] MV02 move_id en double               lignes=     0       4.1 ms


0

**Règle MV03 date invalide.** Date indispensable.

In [22]:
# [STAGING] MV03 date invalide : étape exécutée, chronométrée et journalisée
run_step(con, run_id, "STAGING", "MV03 date invalide", """
INSERT INTO etl_rejects (source_table, raw_rowid, business_key, rule_id, rule_description)
SELECT 'yard_moves', rowid, move_id, 'MV03', 'move_ts illisible'
FROM raw_yard_moves WHERE datetime(move_ts) IS NULL
""")

[STAGING] MV03 date invalide                   lignes=     0       0.9 ms


0

**Règle MV04 equipement inconnu.** Règle FK_EQUIPMENT du sujet.

In [23]:
# [STAGING] MV04 equipement inconnu : étape exécutée, chronométrée et journalisée
run_step(con, run_id, "STAGING", "MV04 equipement inconnu", """
INSERT INTO etl_rejects (source_table, raw_rowid, business_key, rule_id, rule_description)
SELECT 'yard_moves', rowid, move_id, 'MV04', 'equipment_id absent du referentiel'
FROM raw_yard_moves WHERE equipment_id NOT IN (SELECT equipment_id FROM stg_equipment)
""")

[STAGING] MV04 equipement inconnu              lignes=     0       1.3 ms


0

**Règle MV05 zone manquante.** Un mouvement doit avoir un départ et une arrivée.

In [24]:
# [STAGING] MV05 zone manquante : étape exécutée, chronométrée et journalisée
run_step(con, run_id, "STAGING", "MV05 zone manquante", """
INSERT INTO etl_rejects (source_table, raw_rowid, business_key, rule_id, rule_description)
SELECT 'yard_moves', rowid, move_id, 'MV05', 'from_area ou to_area manquante'
FROM raw_yard_moves WHERE from_area IS NULL OR TRIM(from_area) = '' OR to_area IS NULL OR TRIM(to_area) = ''
""")

[STAGING] MV05 zone manquante                  lignes=     0       0.8 ms


0

**Règle MV06 conteneur inconnu.** Règle FK_CONTAINER du sujet.

In [25]:
# [STAGING] MV06 conteneur inconnu : étape exécutée, chronométrée et journalisée
run_step(con, run_id, "STAGING", "MV06 conteneur inconnu", """
INSERT INTO etl_rejects (source_table, raw_rowid, business_key, rule_id, rule_description)
SELECT 'yard_moves', rowid, move_id, 'MV06', 'container_id absent de container_events'
FROM raw_yard_moves WHERE TRIM(container_id) NOT IN (SELECT TRIM(container_id) FROM raw_container_events)
""")

[STAGING] MV06 conteneur inconnu               lignes=     0       6.8 ms


0

### Résultat des règles de rejet

In [26]:
# Nombre de lignes écartées par règle
q('SELECT rule_id, rule_description, COUNT(*) AS nb FROM etl_rejects GROUP BY rule_id, rule_description')

,rule_id,rule_description,nb


**Interprétation.** Le tableau est vide : aucune ligne des données fournies ne viole une règle. Un tableau vide ne prouve pas que les règles fonctionnent : le § 8 le démontre en injectant des erreurs.

**Événements nettoyés.** Seules les lignes **non rejetées** passent : typage, TRIM et zone standardisée. La zone d'origine est conservée dans `yard_area_raw`.

In [27]:
# [STAGING] stg_container_events : étape exécutée, chronométrée et journalisée
run_step(con, run_id, "STAGING", "stg_container_events", """
CREATE TABLE stg_container_events AS
SELECT rowid AS raw_rowid,
       CAST(event_id AS INTEGER)        AS event_id,
       TRIM(container_id)               AS container_id,
       TRIM(customer_id)                AS customer_id,
       TRIM(vessel_id)                  AS vessel_id,
       TRIM(event_type)                 AS event_type,
       datetime(event_ts)               AS event_ts,
       TRIM(yard_area)                  AS yard_area_raw,
       SUBSTR(TRIM(yard_area), 1, 3)    AS yard_area_std,   -- Y08B -> Y08
       TRIM(freight_status)             AS freight_status,
       CAST(gross_weight_kg AS REAL)    AS gross_weight_kg
FROM raw_container_events
WHERE rowid NOT IN (SELECT raw_rowid FROM etl_rejects WHERE source_table = 'container_events')
""")

[STAGING] stg_container_events                 lignes=  9000      12.2 ms


9000

**Mouvements nettoyés.** Même principe : typage et exclusion des lignes rejetées.

In [28]:
# [STAGING] stg_yard_moves : étape exécutée, chronométrée et journalisée
run_step(con, run_id, "STAGING", "stg_yard_moves", """
CREATE TABLE stg_yard_moves AS
SELECT rowid AS raw_rowid,
       CAST(move_id AS INTEGER) AS move_id, TRIM(container_id) AS container_id,
       datetime(move_ts) AS move_ts, TRIM(from_area) AS from_area, TRIM(to_area) AS to_area,
       TRIM(equipment_id) AS equipment_id
FROM raw_yard_moves
WHERE rowid NOT IN (SELECT raw_rowid FROM etl_rejects WHERE source_table = 'yard_moves')
""")

[STAGING] stg_yard_moves                       lignes=  4500       5.3 ms


4500

In [29]:
q('SELECT * FROM stg_container_events LIMIT 5')

,raw_rowid,event_id,container_id,customer_id,vessel_id,event_type,event_ts,yard_area_raw,yard_area_std,freight_status,gross_weight_kg
0,1,1,CONT0003316,C050,V030,MOVE,2026-03-25 01:47:32,Y08B,Y08,FCL,8822.5
1,2,2,CONT0000650,C079,V074,SEAL,2026-03-19 23:42:20,Y02,Y02,FCL,24343.6
2,3,3,CONT0000269,C013,V006,SEAL,2026-03-20 15:15:26,Y04,Y04,FCL,20569.2
3,4,4,CONT0001082,C051,V038,DELIVER,2026-06-18 09:43:51,Y08B,Y08,FCL,30817.3
4,5,5,CONT0002622,C027,V095,DISCHARGE,2026-06-26 09:49:04,Y08,Y08,MTY,25745.9


Comparaison de la zone d'origine et de la zone standardisée :

In [30]:
q("SELECT yard_area_raw, yard_area_std, COUNT(*) AS nb FROM stg_container_events GROUP BY 1, 2 ORDER BY 1")

,yard_area_raw,yard_area_std,nb
0,Y01,Y01,1447
1,Y02,Y02,1489
2,Y03,Y03,1485
3,Y04,Y04,1579
4,Y08,Y08,1519
5,Y08B,Y08,1481


**Interprétation.** Seule `Y08B` est modifiée (rattachée à `Y08`, 1 481 événements). C'est une **hypothèse** à faire valider par le responsable métier ; comme la zone d'origine est conservée, le choix reste réversible.

## 3. Couche CORE (Data Warehouse en étoile)
### Le modèle
Le but du projet (comprendre les volumes, la nature des opérations et les pics d'activité, pour planifier les ressources) détermine les **faits** : ce que l'on veut compter. Les **dimensions** sont les axes selon lesquels on les regarde.

| Fait | Grain (1 ligne =) | Mesures | Question métier |
|---|---|---|---|
| `fact_container_event` | 1 événement conteneur | nombre d'événements, poids | Volumes, opérations, pics (les 6 KPI du sujet) |
| `fact_yard_move` | 1 mouvement de cour | nombre de mouvements, indicateur `is_same_area` | Utilisation de la cour et des équipements |

**Pourquoi deux faits ?** Les deux sources décrivent deux réalités différentes (une opération sur un conteneur, un déplacement dans la cour), avec des dimensions propres. Les fusionner créerait des colonnes vides et fausserait les comptages. Les dimensions `dim_date`, `dim_hour` et `dim_yard_area` sont **partagées** par les deux faits, ce qui permet de les comparer sur les mêmes axes.

`container_id`, `event_id` et `move_id` restent dans les faits comme *dimensions dégénérées* : il n'existe pas de table des conteneurs.

### La construction
Dimensions d'abord, puis faits. Les clés métier (`customer_id`...) sont remplacées par des **clés de substitution** (`customer_key`...).

**Structure de `dim_date`.** Table vide avec une **clé de substitution** (ou la clé de date / d'heure) et les attributs d'analyse.

In [31]:
# [CORE] create dim_date : étape exécutée, chronométrée et journalisée
run_step(con, run_id, "CORE", "create dim_date", """
CREATE TABLE dim_date (date_key INTEGER PRIMARY KEY, full_date TEXT NOT NULL, year INTEGER, month INTEGER, day INTEGER,
    year_month TEXT, week_of_year INTEGER, day_of_week INTEGER, is_weekend INTEGER)
""")

[CORE   ] create dim_date                      lignes=     -       1.7 ms


**Structure de `dim_hour`.** Table vide avec une **clé de substitution** (ou la clé de date / d'heure) et les attributs d'analyse.

In [32]:
# [CORE] create dim_hour : étape exécutée, chronométrée et journalisée
run_step(con, run_id, "CORE", "create dim_hour", """
CREATE TABLE dim_hour (hour_key INTEGER PRIMARY KEY, period TEXT)
""")

[CORE   ] create dim_hour                      lignes=     -       1.9 ms


**Structure de `dim_customer`.** Table vide avec une **clé de substitution** (ou la clé de date / d'heure) et les attributs d'analyse.

In [33]:
# [CORE] create dim_customer : étape exécutée, chronométrée et journalisée
run_step(con, run_id, "CORE", "create dim_customer", """
CREATE TABLE dim_customer (customer_key INTEGER PRIMARY KEY AUTOINCREMENT, customer_id TEXT NOT NULL UNIQUE,
    customer_name TEXT, group_code TEXT, customer_type TEXT)
""")

[CORE   ] create dim_customer                  lignes=     -       1.5 ms


**Structure de `dim_vessel`.** Table vide avec une **clé de substitution** (ou la clé de date / d'heure) et les attributs d'analyse.

In [34]:
# [CORE] create dim_vessel : étape exécutée, chronométrée et journalisée
run_step(con, run_id, "CORE", "create dim_vessel", """
CREATE TABLE dim_vessel (vessel_key INTEGER PRIMARY KEY AUTOINCREMENT, vessel_id TEXT NOT NULL UNIQUE,
    vessel_name TEXT, vessel_type TEXT, service_code TEXT)
""")

[CORE   ] create dim_vessel                    lignes=     -       1.8 ms


**Structure de `dim_yard_area`.** Table vide avec une **clé de substitution** (ou la clé de date / d'heure) et les attributs d'analyse.

In [35]:
# [CORE] create dim_yard_area : étape exécutée, chronométrée et journalisée
run_step(con, run_id, "CORE", "create dim_yard_area", """
CREATE TABLE dim_yard_area (area_key INTEGER PRIMARY KEY AUTOINCREMENT, area_code TEXT NOT NULL UNIQUE,
    parent_area TEXT NOT NULL, is_subarea INTEGER NOT NULL)
""")

[CORE   ] create dim_yard_area                 lignes=     -       1.5 ms


**Structure de `dim_event_type`.** Table vide avec une **clé de substitution** (ou la clé de date / d'heure) et les attributs d'analyse.

In [36]:
# [CORE] create dim_event_type : étape exécutée, chronométrée et journalisée
run_step(con, run_id, "CORE", "create dim_event_type", """
CREATE TABLE dim_event_type (event_type_key INTEGER PRIMARY KEY AUTOINCREMENT, event_type TEXT NOT NULL UNIQUE, event_family TEXT)
""")

[CORE   ] create dim_event_type                lignes=     -       1.9 ms


**Structure de `dim_freight_status`.** Table vide avec une **clé de substitution** (ou la clé de date / d'heure) et les attributs d'analyse.

In [37]:
# [CORE] create dim_freight_status : étape exécutée, chronométrée et journalisée
run_step(con, run_id, "CORE", "create dim_freight_status", """
CREATE TABLE dim_freight_status (freight_key INTEGER PRIMARY KEY AUTOINCREMENT, freight_status TEXT NOT NULL UNIQUE, freight_label TEXT)
""")

[CORE   ] create dim_freight_status            lignes=     -       2.6 ms


**Structure de `dim_equipment`.** Table vide avec une **clé de substitution** (ou la clé de date / d'heure) et les attributs d'analyse.

In [38]:
# [CORE] create dim_equipment : étape exécutée, chronométrée et journalisée
run_step(con, run_id, "CORE", "create dim_equipment", """
CREATE TABLE dim_equipment (equipment_key INTEGER PRIMARY KEY AUTOINCREMENT, equipment_id TEXT NOT NULL UNIQUE,
    equipment_type TEXT, home_area TEXT)
""")

[CORE   ] create dim_equipment                 lignes=     -       1.6 ms


**Chargement de `dim_date`.** Une ligne par jour présent dans les événements ou les mouvements.

In [39]:
# [CORE] load dim_date : étape exécutée, chronométrée et journalisée
run_step(con, run_id, "CORE", "load dim_date", """
INSERT INTO dim_date
SELECT CAST(strftime('%Y%m%d', d) AS INTEGER), d, CAST(strftime('%Y', d) AS INTEGER), CAST(strftime('%m', d) AS INTEGER),
       CAST(strftime('%d', d) AS INTEGER), strftime('%Y-%m', d), CAST(strftime('%W', d) AS INTEGER),
       CAST(strftime('%w', d) AS INTEGER), CASE WHEN strftime('%w', d) IN ('0','6') THEN 1 ELSE 0 END
FROM (SELECT DISTINCT DATE(event_ts) AS d FROM stg_container_events
      UNION SELECT DISTINCT DATE(move_ts) FROM stg_yard_moves)
""")

[CORE   ] load dim_date                        lignes=   301       8.0 ms


301

**Chargement de `dim_hour`.** Les 24 heures, générées par une requête récursive.

In [40]:
# [CORE] load dim_hour : étape exécutée, chronométrée et journalisée
run_step(con, run_id, "CORE", "load dim_hour", """
WITH RECURSIVE h(n) AS (SELECT 0 UNION ALL SELECT n+1 FROM h WHERE n < 23)
INSERT INTO dim_hour
SELECT n, CASE WHEN n < 6 THEN 'Nuit' WHEN n < 12 THEN 'Matin' WHEN n < 18 THEN 'Apres-midi' ELSE 'Soir' END FROM h
""")

[CORE   ] load dim_hour                        lignes=    24       2.4 ms


24

**Chargement de `dim_customer`.** Reprise du référentiel nettoyé, triée pour obtenir des clés stables d'une exécution à l'autre.

In [41]:
# [CORE] load dim_customer : étape exécutée, chronométrée et journalisée
run_step(con, run_id, "CORE", "load dim_customer", """
INSERT INTO dim_customer (customer_id, customer_name, group_code, customer_type)
SELECT customer_id, customer_name, group_code, customer_type FROM stg_customers ORDER BY customer_id
""")

[CORE   ] load dim_customer                    lignes=    80       1.7 ms


80

**Chargement de `dim_vessel`.** Reprise du référentiel nettoyé, triée pour obtenir des clés stables d'une exécution à l'autre.

In [42]:
# [CORE] load dim_vessel : étape exécutée, chronométrée et journalisée
run_step(con, run_id, "CORE", "load dim_vessel", """
INSERT INTO dim_vessel (vessel_id, vessel_name, vessel_type, service_code)
SELECT vessel_id, vessel_name, vessel_type, service_code FROM stg_vessels ORDER BY vessel_id
""")

[CORE   ] load dim_vessel                      lignes=   120       2.0 ms


120

**Chargement de `dim_yard_area`.** Union de toutes les zones. La zone saisie et la zone parente sont gardées : les deux lectures de Y08B restent possibles.

In [43]:
# [CORE] load dim_yard_area : étape exécutée, chronométrée et journalisée
run_step(con, run_id, "CORE", "load dim_yard_area", """
INSERT INTO dim_yard_area (area_code, parent_area, is_subarea)
SELECT a, SUBSTR(a, 1, 3), CASE WHEN LENGTH(a) > 3 THEN 1 ELSE 0 END   -- regle : suffixe lettre = sous-zone
FROM (SELECT yard_area_raw AS a FROM stg_container_events
      UNION SELECT from_area FROM stg_yard_moves UNION SELECT to_area FROM stg_yard_moves) ORDER BY a
""")

[CORE   ] load dim_yard_area                   lignes=     6       6.9 ms


6

**Chargement de `dim_event_type`.** Les types sont regroupés en familles d'opérations (hypothèse métier à valider).

In [44]:
# [CORE] load dim_event_type : étape exécutée, chronométrée et journalisée
run_step(con, run_id, "CORE", "load dim_event_type", """
INSERT INTO dim_event_type (event_type, event_family)
SELECT DISTINCT event_type,
       CASE event_type WHEN 'DISCHARGE' THEN 'Operations navire' WHEN 'LOAD' THEN 'Operations navire'
                       WHEN 'RECEIVE' THEN 'Gate' WHEN 'DELIVER' THEN 'Gate' WHEN 'MOVE' THEN 'Cour'
                       WHEN 'STUFF' THEN 'Manutention' WHEN 'SEAL' THEN 'Manutention' ELSE 'Autre' END
FROM stg_container_events ORDER BY event_type
""")

[CORE   ] load dim_event_type                  lignes=     7       3.4 ms


7

**Chargement de `dim_freight_status`.** Libellés lisibles : FCL = plein, MTY = vide.

In [45]:
# [CORE] load dim_freight_status : étape exécutée, chronométrée et journalisée
run_step(con, run_id, "CORE", "load dim_freight_status", """
INSERT INTO dim_freight_status (freight_status, freight_label)
SELECT DISTINCT freight_status, CASE freight_status WHEN 'FCL' THEN 'Plein' WHEN 'MTY' THEN 'Vide' ELSE 'Inconnu' END
FROM stg_container_events ORDER BY freight_status
""")

[CORE   ] load dim_freight_status              lignes=     2       3.1 ms


2

**Chargement de `dim_equipment`.** Reprise du référentiel nettoyé, triée pour obtenir des clés stables d'une exécution à l'autre.

In [46]:
# [CORE] load dim_equipment : étape exécutée, chronométrée et journalisée
run_step(con, run_id, "CORE", "load dim_equipment", """
INSERT INTO dim_equipment (equipment_id, equipment_type, home_area)
SELECT equipment_id, equipment_type, home_area FROM stg_equipment ORDER BY equipment_id
""")

[CORE   ] load dim_equipment                   lignes=    40       1.7 ms


40

**Structure de `fact_container_event`.** Table de faits : une clé par dimension, les mesures et les identifiants d'origine (dimensions dégénérées).

In [47]:
# [CORE] create fact_container_event : étape exécutée, chronométrée et journalisée
run_step(con, run_id, "CORE", "create fact_container_event", """
CREATE TABLE fact_container_event (
    event_id INTEGER PRIMARY KEY, container_id TEXT NOT NULL,
    date_key INTEGER NOT NULL REFERENCES dim_date(date_key), hour_key INTEGER NOT NULL REFERENCES dim_hour(hour_key),
    customer_key INTEGER NOT NULL REFERENCES dim_customer(customer_key), vessel_key INTEGER NOT NULL REFERENCES dim_vessel(vessel_key),
    area_key INTEGER NOT NULL REFERENCES dim_yard_area(area_key), event_type_key INTEGER NOT NULL REFERENCES dim_event_type(event_type_key),
    freight_key INTEGER NOT NULL REFERENCES dim_freight_status(freight_key),
    event_ts TEXT NOT NULL, event_count INTEGER NOT NULL DEFAULT 1, gross_weight_kg REAL)
""")

[CORE   ] create fact_container_event          lignes=     -       1.5 ms


**Structure de `fact_yard_move`.** Table de faits : une clé par dimension, les mesures et les identifiants d'origine (dimensions dégénérées).

In [48]:
# [CORE] create fact_yard_move : étape exécutée, chronométrée et journalisée
run_step(con, run_id, "CORE", "create fact_yard_move", """
CREATE TABLE fact_yard_move (
    move_id INTEGER PRIMARY KEY, container_id TEXT NOT NULL,
    date_key INTEGER NOT NULL REFERENCES dim_date(date_key), hour_key INTEGER NOT NULL REFERENCES dim_hour(hour_key),
    from_area_key INTEGER NOT NULL REFERENCES dim_yard_area(area_key), to_area_key INTEGER NOT NULL REFERENCES dim_yard_area(area_key),
    equipment_key INTEGER NOT NULL REFERENCES dim_equipment(equipment_key),
    move_ts TEXT NOT NULL, move_count INTEGER NOT NULL DEFAULT 1, is_same_area INTEGER NOT NULL)
""")

[CORE   ] create fact_yard_move                lignes=     -       1.3 ms


**Chargement de `fact_container_event`.** Les codes métier sont remplacés par les clés de substitution, au moyen de jointures sur les dimensions.

In [49]:
# [CORE] load fact_container_event : étape exécutée, chronométrée et journalisée
run_step(con, run_id, "CORE", "load fact_container_event", """
INSERT INTO fact_container_event
SELECT e.event_id, e.container_id, CAST(strftime('%Y%m%d', e.event_ts) AS INTEGER), CAST(strftime('%H', e.event_ts) AS INTEGER),
       c.customer_key, v.vessel_key, a.area_key, t.event_type_key, f.freight_key, e.event_ts, 1, e.gross_weight_kg
FROM stg_container_events e
JOIN dim_customer c       ON c.customer_id = e.customer_id
JOIN dim_vessel v         ON v.vessel_id = e.vessel_id
JOIN dim_yard_area a      ON a.area_code = e.yard_area_raw
JOIN dim_event_type t     ON t.event_type = e.event_type
JOIN dim_freight_status f ON f.freight_status = e.freight_status
""")

[CORE   ] load fact_container_event            lignes=  9000      17.2 ms


9000

**Chargement de `fact_yard_move`.** Les codes métier sont remplacés par les clés de substitution, au moyen de jointures sur les dimensions.

In [50]:
# [CORE] load fact_yard_move : étape exécutée, chronométrée et journalisée
run_step(con, run_id, "CORE", "load fact_yard_move", """
INSERT INTO fact_yard_move
SELECT m.move_id, m.container_id, CAST(strftime('%Y%m%d', m.move_ts) AS INTEGER), CAST(strftime('%H', m.move_ts) AS INTEGER),
       af.area_key, at.area_key, q.equipment_key, m.move_ts, 1, CASE WHEN m.from_area = m.to_area THEN 1 ELSE 0 END
FROM stg_yard_moves m
JOIN dim_yard_area af ON af.area_code = m.from_area
JOIN dim_yard_area at ON at.area_code = m.to_area
JOIN dim_equipment q  ON q.equipment_id = m.equipment_id
""")

[CORE   ] load fact_yard_move                  lignes=  4500       8.3 ms


4500

Aperçu du fait principal, avec ses clés de substitution :

In [51]:
q('SELECT * FROM fact_container_event LIMIT 5')

,event_id,container_id,date_key,hour_key,customer_key,vessel_key,area_key,event_type_key,freight_key,event_ts,event_count,gross_weight_kg
0,1,CONT0003316,20260325,1,50,30,6,4,1,2026-03-25 01:47:32,1,8822.5
1,2,CONT0000650,20260319,23,79,74,2,6,1,2026-03-19 23:42:20,1,24343.6
2,3,CONT0000269,20260320,15,13,6,4,6,1,2026-03-20 15:15:26,1,20569.2
3,4,CONT0001082,20260618,9,51,38,6,1,1,2026-06-18 09:43:51,1,30817.3
4,5,CONT0002622,20260626,9,27,95,5,2,2,2026-06-26 09:49:04,1,25745.9


In [52]:
# Taille de chaque table du CORE
q(" UNION ALL ".join(f"SELECT '{t}' AS table_core, COUNT(*) AS lignes FROM {t}" for t in
    ["fact_container_event", "fact_yard_move", "dim_date", "dim_hour", "dim_customer", "dim_vessel",
     "dim_yard_area", "dim_event_type", "dim_freight_status", "dim_equipment"]))

,table_core,lignes
0,fact_container_event,9000
1,fact_yard_move,4500
2,dim_date,301
3,dim_hour,24
4,dim_customer,80
5,dim_vessel,120
6,dim_yard_area,6
7,dim_event_type,7
8,dim_freight_status,2
9,dim_equipment,40


**Interprétation.** Les deux faits gardent le volume des sources (9 000 et 4 500 lignes) : les jointures n'ont ni perdu ni dupliqué de ligne. `dim_date` couvre 301 jours et `dim_yard_area` contient 6 zones, dont la sous-zone Y08B.

## 4. Couche MART (une table par KPI)
Tables prêtes à être lues par l'EDA ou par un outil de tableau de bord.

**`mart_kpi_events_per_day`.** **KPI 1.** Grain : un jour. Événements, conteneurs distincts et tonnes.

In [53]:
# [MART] mart_kpi_events_per_day : étape exécutée, chronométrée et journalisée
run_step(con, run_id, "MART", "mart_kpi_events_per_day", """
CREATE TABLE mart_kpi_events_per_day AS
SELECT d.full_date, d.year_month, COUNT(*) AS nb_evenements, COUNT(DISTINCT f.container_id) AS nb_conteneurs,
       ROUND(SUM(f.gross_weight_kg)/1000.0, 1) AS tonnes
FROM fact_container_event f JOIN dim_date d ON d.date_key = f.date_key
GROUP BY d.full_date, d.year_month
""")

[MART   ] mart_kpi_events_per_day              lignes=   301       9.1 ms


301

In [54]:
# Contrôle visuel du résultat
q("SELECT * FROM mart_kpi_events_per_day ORDER BY nb_evenements DESC LIMIT 5")

,full_date,year_month,nb_evenements,nb_conteneurs,tonnes
0,2026-06-20,2026-06,50,50,889.9
1,2026-05-30,2026-05,47,46,818.9
2,2026-06-11,2026-06,45,45,843.3
3,2026-07-04,2026-07,44,44,813.2
4,2026-05-05,2026-05,43,43,922.5


**`mart_kpi_move_by_area`.** **KPI 2.** Grain : une zone. Seuls les événements de type MOVE.

In [55]:
# [MART] mart_kpi_move_by_area : étape exécutée, chronométrée et journalisée
run_step(con, run_id, "MART", "mart_kpi_move_by_area", """
CREATE TABLE mart_kpi_move_by_area AS
SELECT a.area_code, a.parent_area, a.is_subarea, COUNT(*) AS nb_move
FROM fact_container_event f
JOIN dim_event_type t ON t.event_type_key = f.event_type_key
JOIN dim_yard_area a  ON a.area_key = f.area_key
WHERE t.event_type = 'MOVE'
GROUP BY a.area_code, a.parent_area, a.is_subarea
""")

[MART   ] mart_kpi_move_by_area                lignes=     6       3.2 ms


6

In [56]:
# Contrôle visuel du résultat
q("SELECT * FROM mart_kpi_move_by_area ORDER BY nb_move DESC")

,area_code,parent_area,is_subarea,nb_move
0,Y04,Y04,0,538
1,Y08,Y08,0,490
2,Y03,Y03,0,460
3,Y08B,Y08,1,456
4,Y01,Y01,0,453
5,Y02,Y02,0,443


**`mart_kpi_vessel`.** **KPI 3 (navire).** Grain : un navire.

In [57]:
# [MART] mart_kpi_vessel : étape exécutée, chronométrée et journalisée
run_step(con, run_id, "MART", "mart_kpi_vessel", """
CREATE TABLE mart_kpi_vessel AS
SELECT v.vessel_id, v.vessel_name, v.vessel_type, v.service_code, COUNT(*) AS nb_evenements,
       COUNT(DISTINCT f.container_id) AS nb_conteneurs, ROUND(SUM(f.gross_weight_kg)/1000.0, 1) AS tonnes
FROM fact_container_event f JOIN dim_vessel v ON v.vessel_key = f.vessel_key
GROUP BY v.vessel_id, v.vessel_name, v.vessel_type, v.service_code
""")

[MART   ] mart_kpi_vessel                      lignes=   120      10.5 ms


120

In [58]:
# Contrôle visuel du résultat
q("SELECT * FROM mart_kpi_vessel ORDER BY nb_evenements DESC LIMIT 5")

,vessel_id,vessel_name,vessel_type,service_code,nb_evenements,nb_conteneurs,tonnes
0,V020,VESSEL_020,Container,REDSEA,100,99,1836.2
1,V039,VESSEL_039,Container,DJJIB,100,97,1845.5
2,V068,VESSEL_068,Container,GULF,100,100,1915.7
3,V090,VESSEL_090,Feeder,GULF,97,97,1865.0
4,V120,VESSEL_120,Feeder,HORN,93,93,1845.9


**`mart_kpi_service`.** **KPI 3 (service).** Construit à partir de `mart_kpi_vessel`. Les tonnes additionnent des valeurs déjà arrondies.

In [59]:
# [MART] mart_kpi_service : étape exécutée, chronométrée et journalisée
run_step(con, run_id, "MART", "mart_kpi_service", """
CREATE TABLE mart_kpi_service AS
SELECT service_code, COUNT(*) AS nb_navires, SUM(nb_evenements) AS nb_evenements,
       ROUND(1.0*SUM(nb_evenements)/COUNT(*), 1) AS evenements_par_navire,
       ROUND(SUM(tonnes), 0) AS tonnes,
       ROUND(100.0*SUM(nb_evenements)/(SELECT SUM(nb_evenements) FROM mart_kpi_vessel), 1) AS pct_evenements
FROM mart_kpi_vessel GROUP BY service_code
""")

[MART   ] mart_kpi_service                     lignes=     5       2.3 ms


5

In [60]:
# Contrôle visuel du résultat
q("SELECT * FROM mart_kpi_service ORDER BY nb_evenements DESC")

,service_code,nb_navires,nb_evenements,evenements_par_navire,tonnes,pct_evenements
0,GULF,32,2469,77.2,46010.0,27.4
1,REDSEA,26,1959,75.3,36071.0,21.8
2,DJJIB,24,1801,75.0,33672.0,20.0
3,HORN,21,1550,73.8,29125.0,17.2
4,ASIA,17,1221,71.8,22482.0,13.6


**`mart_kpi_customer`.** **KPI 4.** Grain : un client.

In [61]:
# [MART] mart_kpi_customer : étape exécutée, chronométrée et journalisée
run_step(con, run_id, "MART", "mart_kpi_customer", """
CREATE TABLE mart_kpi_customer AS
SELECT c.customer_id, c.customer_name, c.customer_type, c.group_code, COUNT(*) AS nb_evenements,
       COUNT(DISTINCT f.container_id) AS nb_conteneurs, ROUND(SUM(f.gross_weight_kg)/1000.0, 1) AS tonnes,
       ROUND(100.0*SUM(fs.freight_status = 'FCL')/COUNT(*), 1) AS pct_fcl
FROM fact_container_event f
JOIN dim_customer c        ON c.customer_key = f.customer_key
JOIN dim_freight_status fs ON fs.freight_key = f.freight_key
GROUP BY c.customer_id, c.customer_name, c.customer_type, c.group_code
""")

[MART   ] mart_kpi_customer                    lignes=    80      10.9 ms


80

In [62]:
# Contrôle visuel du résultat
q("SELECT * FROM mart_kpi_customer ORDER BY nb_evenements DESC LIMIT 5")

,customer_id,customer_name,customer_type,group_code,nb_evenements,nb_conteneurs,tonnes,pct_fcl
0,C067,Customer_067,Forwarder,ETH_MTL,136,135,2513.7,76.5
1,C029,Customer_029,Importer,FZ,134,131,2415.6,79.1
2,C077,Customer_077,Importer,DJ,134,132,2523.6,80.6
3,C044,Customer_044,Importer,ETH,127,126,2361.2,81.1
4,C071,Customer_071,Importer,FZ,127,125,2365.9,69.3


**`mart_kpi_event_type`.** **KPI 5.** Grain : un type d'événement. Poids moyen et part de FCL.

In [63]:
# [MART] mart_kpi_event_type : étape exécutée, chronométrée et journalisée
run_step(con, run_id, "MART", "mart_kpi_event_type", """
CREATE TABLE mart_kpi_event_type AS
SELECT t.event_type, t.event_family, COUNT(*) AS nb_evenements, ROUND(AVG(f.gross_weight_kg), 0) AS poids_moyen_kg,
       ROUND(100.0*SUM(fs.freight_status = 'FCL')/COUNT(*), 1) AS pct_fcl
FROM fact_container_event f
JOIN dim_event_type t      ON t.event_type_key = f.event_type_key
JOIN dim_freight_status fs ON fs.freight_key = f.freight_key
GROUP BY t.event_type, t.event_family
""")

[MART   ] mart_kpi_event_type                  lignes=     7       6.0 ms


7

In [64]:
# Contrôle visuel du résultat
q("SELECT * FROM mart_kpi_event_type ORDER BY nb_evenements DESC")

,event_type,event_family,nb_evenements,poids_moyen_kg,pct_fcl
0,MOVE,Cour,2840,18488.0,75.6
1,DISCHARGE,Operations navire,1413,18692.0,76.1
2,LOAD,Operations navire,1332,18692.0,75.9
3,DELIVER,Gate,1062,18444.0,77.0
4,RECEIVE,Gate,932,18750.0,75.6
5,STUFF,Manutention,714,18477.0,76.2
6,SEAL,Manutention,707,18797.0,74.0


**`mart_kpi_freight`.** **KPI 6.** Grain : un statut FCL ou MTY.

In [65]:
# [MART] mart_kpi_freight : étape exécutée, chronométrée et journalisée
run_step(con, run_id, "MART", "mart_kpi_freight", """
CREATE TABLE mart_kpi_freight AS
SELECT fs.freight_status, fs.freight_label, COUNT(*) AS nb_evenements,
       ROUND(100.0*COUNT(*)/(SELECT COUNT(*) FROM fact_container_event), 1) AS pct_evenements,
       ROUND(AVG(f.gross_weight_kg), 0) AS poids_moyen_kg
FROM fact_container_event f JOIN dim_freight_status fs ON fs.freight_key = f.freight_key
GROUP BY fs.freight_status, fs.freight_label
""")

[MART   ] mart_kpi_freight                     lignes=     2       5.1 ms


2

In [66]:
# Contrôle visuel du résultat
q("SELECT * FROM mart_kpi_freight")

,freight_status,freight_label,nb_evenements,pct_evenements,poids_moyen_kg
0,FCL,Plein,6824,75.8,18582.0
1,MTY,Vide,2176,24.2,18639.0


**Interprétation.** Les six indicateurs du sujet sont matérialisés dans sept tables (le KPI 3 en a deux : navire et service). Le jour le plus chargé compte 50 événements, Y04 est la première zone pour les MOVE (538), GULF le premier service (2 469) et 75,8 % des événements concernent des conteneurs pleins. L'analyse détaillée est dans le notebook 02.

## 5. Contrôles qualité (15 contrôles du pipeline + les 17 règles du prof)
Chaque contrôle compare une valeur **observée** à une valeur **attendue**.
- `PASS` : conforme · `FAIL` : anomalie bloquante · `WARN` : anomalie métier signalée mais conservée · `INFO` : simple information.

### 5.1 La fonction `controle`
Elle exécute une requête, compare le résultat à la valeur attendue, affiche le statut, et l'enregistre dans la table `etl_quality_checks`.

In [67]:
# Un contrôle = une requête, une valeur attendue et une gravité
def init_quality_table(con):
    con.execute("DROP TABLE IF EXISTS etl_quality_checks")
    con.execute("""CREATE TABLE etl_quality_checks (check_id TEXT, layer TEXT, description TEXT,
                   observed REAL, expected REAL, status TEXT)""")
    con.commit()

def controle(con, check_id, layer, description, sql, expected=0, severity="FAIL", register=True, verbose=True):
    observed = con.execute(sql).fetchone()[0]
    status = "PASS" if observed == expected else severity
    con.execute("INSERT INTO etl_quality_checks VALUES (?,?,?,?,?,?)",
                (check_id, layer, description, observed, expected, status))
    con.commit()
    if register and not any(c[0] == check_id for c in CHECKS):
        CHECKS.append((check_id, layer, description, sql, expected, severity))
    if verbose:
        print(f"{status:<5} {check_id} [{layer}] {description} -> observé = {observed:g}")
    return status

init_quality_table(con)

### 5.2 Les 15 contrôles du projet

**Q01** · Volume : 1 si raw_container_events est vide (attendu 0)

In [68]:
# Résultat comparé à la valeur attendue, puis enregistré dans etl_quality_checks
controle(con, "Q01", "RAW", "Volume : 1 si raw_container_events est vide (attendu 0)", """
SELECT CASE WHEN COUNT(*) > 0 THEN 0 ELSE 1 END FROM raw_container_events
""", 0, "FAIL")

PASS  Q01 [RAW] Volume : 1 si raw_container_events est vide (attendu 0) -> observé = 0


'PASS'

**Q02** · Unicite de event_id dans fact_container_event

In [69]:
# Résultat comparé à la valeur attendue, puis enregistré dans etl_quality_checks
controle(con, "Q02", "CORE", "Unicite de event_id dans fact_container_event", """
SELECT COUNT(*) - COUNT(DISTINCT event_id) FROM fact_container_event
""", 0, "FAIL")

PASS  Q02 [CORE] Unicite de event_id dans fact_container_event -> observé = 0


'PASS'

**Q03** · Unicite de move_id dans fact_yard_move

In [70]:
# Résultat comparé à la valeur attendue, puis enregistré dans etl_quality_checks
controle(con, "Q03", "CORE", "Unicite de move_id dans fact_yard_move", """
SELECT COUNT(*) - COUNT(DISTINCT move_id) FROM fact_yard_move
""", 0, "FAIL")

PASS  Q03 [CORE] Unicite de move_id dans fact_yard_move -> observé = 0


'PASS'

**Q04** · Valeurs nulles dans les colonnes cles de stg_container_events

In [71]:
# Résultat comparé à la valeur attendue, puis enregistré dans etl_quality_checks
controle(con, "Q04", "STAGING", "Valeurs nulles dans les colonnes cles de stg_container_events", """
SELECT COUNT(*) FROM stg_container_events WHERE event_id IS NULL OR container_id IS NULL OR customer_id IS NULL
     OR vessel_id IS NULL OR event_type IS NULL OR event_ts IS NULL OR yard_area_raw IS NULL OR gross_weight_kg IS NULL
""", 0, "FAIL")

PASS  Q04 [STAGING] Valeurs nulles dans les colonnes cles de stg_container_events -> observé = 0


'PASS'

**Q05** · Integrite referentielle : cles de fact_container_event absentes des dimensions

In [72]:
# Résultat comparé à la valeur attendue, puis enregistré dans etl_quality_checks
controle(con, "Q05", "CORE", "Integrite referentielle : cles de fact_container_event absentes des dimensions", """
SELECT (SELECT COUNT(*) FROM fact_container_event WHERE customer_key NOT IN (SELECT customer_key FROM dim_customer))
          + (SELECT COUNT(*) FROM fact_container_event WHERE vessel_key NOT IN (SELECT vessel_key FROM dim_vessel))
          + (SELECT COUNT(*) FROM fact_container_event WHERE area_key NOT IN (SELECT area_key FROM dim_yard_area))
          + (SELECT COUNT(*) FROM fact_container_event WHERE event_type_key NOT IN (SELECT event_type_key FROM dim_event_type))
          + (SELECT COUNT(*) FROM fact_container_event WHERE freight_key NOT IN (SELECT freight_key FROM dim_freight_status))
          + (SELECT COUNT(*) FROM fact_container_event WHERE date_key NOT IN (SELECT date_key FROM dim_date))
""", 0, "FAIL")

PASS  Q05 [CORE] Integrite referentielle : cles de fact_container_event absentes des dimensions -> observé = 0


'PASS'

**Q06** · Integrite referentielle : cles de fact_yard_move absentes des dimensions

In [73]:
# Résultat comparé à la valeur attendue, puis enregistré dans etl_quality_checks
controle(con, "Q06", "CORE", "Integrite referentielle : cles de fact_yard_move absentes des dimensions", """
SELECT (SELECT COUNT(*) FROM fact_yard_move WHERE from_area_key NOT IN (SELECT area_key FROM dim_yard_area))
          + (SELECT COUNT(*) FROM fact_yard_move WHERE to_area_key NOT IN (SELECT area_key FROM dim_yard_area))
          + (SELECT COUNT(*) FROM fact_yard_move WHERE equipment_key NOT IN (SELECT equipment_key FROM dim_equipment))
          + (SELECT COUNT(*) FROM fact_yard_move WHERE date_key NOT IN (SELECT date_key FROM dim_date))
""", 0, "FAIL")

PASS  Q06 [CORE] Integrite referentielle : cles de fact_yard_move absentes des dimensions -> observé = 0


'PASS'

**Q07** · Domaine des valeurs : event_type et freight_status hors referentiel

In [74]:
# Résultat comparé à la valeur attendue, puis enregistré dans etl_quality_checks
controle(con, "Q07", "STAGING", "Domaine des valeurs : event_type et freight_status hors referentiel", """
SELECT COUNT(*) FROM stg_container_events
      WHERE event_type NOT IN ('DISCHARGE','LOAD','MOVE','DELIVER','RECEIVE','STUFF','SEAL') OR freight_status NOT IN ('FCL','MTY')
""", 0, "FAIL")

PASS  Q07 [STAGING] Domaine des valeurs : event_type et freight_status hors referentiel -> observé = 0


'PASS'

**Q08** · Plage du poids : valeurs hors 0..40000 kg dans fact_container_event

In [75]:
# Résultat comparé à la valeur attendue, puis enregistré dans etl_quality_checks
controle(con, "Q08", "CORE", "Plage du poids : valeurs hors 0..40000 kg dans fact_container_event", """
SELECT COUNT(*) FROM fact_container_event WHERE gross_weight_kg <= 0 OR gross_weight_kg > 40000
""", 0, "FAIL")

PASS  Q08 [CORE] Plage du poids : valeurs hors 0..40000 kg dans fact_container_event -> observé = 0


'PASS'

**Q09** · Reconciliation evenements : RAW - (CORE + rejets)

In [76]:
# Résultat comparé à la valeur attendue, puis enregistré dans etl_quality_checks
controle(con, "Q09", "CORE", "Reconciliation evenements : RAW - (CORE + rejets)", """
SELECT (SELECT COUNT(*) FROM raw_container_events) - (SELECT COUNT(*) FROM fact_container_event)
          - (SELECT COUNT(DISTINCT raw_rowid) FROM etl_rejects WHERE source_table = 'container_events')
""", 0, "FAIL")

PASS  Q09 [CORE] Reconciliation evenements : RAW - (CORE + rejets) -> observé = 0


'PASS'

**Q10** · Reconciliation mouvements : RAW - (CORE + rejets)

In [77]:
# Résultat comparé à la valeur attendue, puis enregistré dans etl_quality_checks
controle(con, "Q10", "CORE", "Reconciliation mouvements : RAW - (CORE + rejets)", """
SELECT (SELECT COUNT(*) FROM raw_yard_moves) - (SELECT COUNT(*) FROM fact_yard_move)
          - (SELECT COUNT(DISTINCT raw_rowid) FROM etl_rejects WHERE source_table = 'yard_moves')
""", 0, "FAIL")

PASS  Q10 [CORE] Reconciliation mouvements : RAW - (CORE + rejets) -> observé = 0


'PASS'

**Q11** · Conservation de la mesure : poids total STAGING - poids total CORE (kg)

In [78]:
# Résultat comparé à la valeur attendue, puis enregistré dans etl_quality_checks
controle(con, "Q11", "CORE", "Conservation de la mesure : poids total STAGING - poids total CORE (kg)", """
SELECT ROUND((SELECT SUM(gross_weight_kg) FROM stg_container_events) - (SELECT SUM(gross_weight_kg) FROM fact_container_event), 3)
""", 0, "FAIL")

PASS  Q11 [CORE] Conservation de la mesure : poids total STAGING - poids total CORE (kg) -> observé = 0


'PASS'

**Q12** · Coherence MART : somme des evenements par jour - lignes de fact_container_event

In [79]:
# Résultat comparé à la valeur attendue, puis enregistré dans etl_quality_checks
controle(con, "Q12", "MART", "Coherence MART : somme des evenements par jour - lignes de fact_container_event", """
SELECT (SELECT SUM(nb_evenements) FROM mart_kpi_events_per_day) - (SELECT COUNT(*) FROM fact_container_event)
""", 0, "FAIL")

PASS  Q12 [MART] Coherence MART : somme des evenements par jour - lignes de fact_container_event -> observé = 0


'PASS'

**Q13** · Coherence metier : mouvements de cour avec from_area = to_area (signales, conserves)

In [80]:
# Résultat comparé à la valeur attendue, puis enregistré dans etl_quality_checks
controle(con, "Q13", "CORE", "Coherence metier : mouvements de cour avec from_area = to_area (signales, conserves)", """
SELECT COUNT(*) FROM fact_yard_move WHERE is_same_area = 1
""", 0, "WARN")

WARN  Q13 [CORE] Coherence metier : mouvements de cour avec from_area = to_area (signales, conserves) -> observé = 712


'WARN'

**Q14** · Integrite : mouvements portant sur un conteneur absent des evenements (regle FK_CONTAINER)

In [81]:
# Résultat comparé à la valeur attendue, puis enregistré dans etl_quality_checks
controle(con, "Q14", "CORE", "Integrite : mouvements portant sur un conteneur absent des evenements (regle FK_CONTAINER)", """
SELECT COUNT(*) FROM fact_yard_move WHERE container_id NOT IN (SELECT container_id FROM fact_container_event)
""", 0, "FAIL")

PASS  Q14 [CORE] Integrite : mouvements portant sur un conteneur absent des evenements (regle FK_CONTAINER) -> observé = 0


'PASS'

**Q15** · Standardisation : evenements dont la zone a ete rattachee a sa zone parente (Y08B -> Y08)

In [82]:
# Résultat comparé à la valeur attendue, puis enregistré dans etl_quality_checks
controle(con, "Q15", "STAGING", "Standardisation : evenements dont la zone a ete rattachee a sa zone parente (Y08B -> Y08)", """
SELECT COUNT(*) FROM stg_container_events WHERE yard_area_raw <> yard_area_std
""", 0, "INFO")

INFO  Q15 [STAGING] Standardisation : evenements dont la zone a ete rattachee a sa zone parente (Y08B -> Y08) -> observé = 1481


'INFO'

### 5.3 Les 17 règles qualité du prof
Le fichier `data_quality_rules.csv` demande, pour **chacune des 5 tables**, trois règles (`ROW_COUNT`, `DUPLICATE_CHECK`, `NULL_RATE`), plus deux règles de clés étrangères sur les mouvements de cour (`FK_CONTAINER`, `FK_EQUIPMENT`).
Ces contrôles portent sur la donnée **source (RAW)**, avant tout nettoyage. Gravité du prof : `HIGH` → `FAIL`, `MEDIUM` → `WARN`.

**ROW_COUNT** · la table n'est pas vide (gravité HIGH)

In [83]:
TABLES_CLES = {"customers": "customer_id", "vessel_calls": "vessel_id", "container_events": "event_id",
              "yard_moves": "move_id", "equipment": "equipment_id"}

for t in TABLES_CLES:
    controle(con, f"DQ_ROW_COUNT_{t}", "RAW", f"ROW_COUNT {t} : table vide ? (0 = non vide)",
             f"SELECT CASE WHEN COUNT(*) > 0 THEN 0 ELSE 1 END FROM raw_{t}", 0, "FAIL")

PASS  DQ_ROW_COUNT_customers [RAW] ROW_COUNT customers : table vide ? (0 = non vide) -> observé = 0
PASS  DQ_ROW_COUNT_vessel_calls [RAW] ROW_COUNT vessel_calls : table vide ? (0 = non vide) -> observé = 0
PASS  DQ_ROW_COUNT_container_events [RAW] ROW_COUNT container_events : table vide ? (0 = non vide) -> observé = 0
PASS  DQ_ROW_COUNT_yard_moves [RAW] ROW_COUNT yard_moves : table vide ? (0 = non vide) -> observé = 0
PASS  DQ_ROW_COUNT_equipment [RAW] ROW_COUNT equipment : table vide ? (0 = non vide) -> observé = 0


Les volumes attendus, pour expliquer l'« évolution » demandée par la règle :

In [84]:
q('''
SELECT 'customers' AS table_source, COUNT(*) AS nb_lignes FROM raw_customers
UNION ALL SELECT 'vessel_calls', COUNT(*) FROM raw_vessel_calls
UNION ALL SELECT 'container_events', COUNT(*) FROM raw_container_events
UNION ALL SELECT 'yard_moves', COUNT(*) FROM raw_yard_moves
UNION ALL SELECT 'equipment', COUNT(*) FROM raw_equipment
''')

,table_source,nb_lignes
0,customers,80
1,vessel_calls,120
2,container_events,9000
3,yard_moves,4500
4,equipment,40


**DUPLICATE_CHECK** · la clé métier est unique (gravité HIGH)

In [85]:
for t, cle in TABLES_CLES.items():
    controle(con, f"DQ_DUPLICATE_CHECK_{t}", "RAW", f"DUPLICATE_CHECK {t} : doublons sur {cle}",
             f"SELECT COUNT(*) - COUNT(DISTINCT TRIM({cle})) FROM raw_{t}", 0, "FAIL")

PASS  DQ_DUPLICATE_CHECK_customers [RAW] DUPLICATE_CHECK customers : doublons sur customer_id -> observé = 0
PASS  DQ_DUPLICATE_CHECK_vessel_calls [RAW] DUPLICATE_CHECK vessel_calls : doublons sur vessel_id -> observé = 0
PASS  DQ_DUPLICATE_CHECK_container_events [RAW] DUPLICATE_CHECK container_events : doublons sur event_id -> observé = 0
PASS  DQ_DUPLICATE_CHECK_yard_moves [RAW] DUPLICATE_CHECK yard_moves : doublons sur move_id -> observé = 0
PASS  DQ_DUPLICATE_CHECK_equipment [RAW] DUPLICATE_CHECK equipment : doublons sur equipment_id -> observé = 0


**NULL_RATE** · pourcentage de cellules vides dans la table (gravité MEDIUM)

In [86]:
for t in TABLES_CLES:
    cols = [r[1] for r in con.execute(f"PRAGMA table_info(raw_{t})") if not r[1].startswith("_")]
    vides = " + ".join(f"SUM(CASE WHEN {c} IS NULL OR TRIM({c}) = '' THEN 1 ELSE 0 END)" for c in cols)
    sql = f"SELECT ROUND(100.0 * ({vides}) / (COUNT(*) * {len(cols)}), 2) FROM raw_{t}"
    controle(con, f"DQ_NULL_RATE_{t}", "RAW", f"NULL_RATE {t} : % de cellules vides", sql, 0, "WARN")

PASS  DQ_NULL_RATE_customers [RAW] NULL_RATE customers : % de cellules vides -> observé = 0
PASS  DQ_NULL_RATE_vessel_calls [RAW] NULL_RATE vessel_calls : % de cellules vides -> observé = 0


PASS  DQ_NULL_RATE_container_events [RAW] NULL_RATE container_events : % de cellules vides -> observé = 0
PASS  DQ_NULL_RATE_yard_moves [RAW] NULL_RATE yard_moves : % de cellules vides -> observé = 0
PASS  DQ_NULL_RATE_equipment [RAW] NULL_RATE equipment : % de cellules vides -> observé = 0


**FK_CONTAINER et FK_EQUIPMENT** · les clés des mouvements existent dans les tables de référence (gravité HIGH)

In [87]:
controle(con, "FK_CONTAINER", "RAW", "FK_CONTAINER : mouvements dont le container_id est absent de container_events",
         "SELECT COUNT(*) FROM raw_yard_moves WHERE TRIM(container_id) NOT IN (SELECT TRIM(container_id) FROM raw_container_events)", 0, "FAIL")

controle(con, "FK_EQUIPMENT", "RAW", "FK_EQUIPMENT : mouvements dont l'equipment_id est absent de equipment",
         "SELECT COUNT(*) FROM raw_yard_moves WHERE TRIM(equipment_id) NOT IN (SELECT TRIM(equipment_id) FROM raw_equipment)", 0, "FAIL")

PASS  FK_CONTAINER [RAW] FK_CONTAINER : mouvements dont le container_id est absent de container_events -> observé = 0
PASS  FK_EQUIPMENT [RAW] FK_EQUIPMENT : mouvements dont l'equipment_id est absent de equipment -> observé = 0


'PASS'

### 5.4 Synthèse

In [88]:
q('SELECT status, COUNT(*) AS nb FROM etl_quality_checks GROUP BY status ORDER BY status')

,status,nb
0,INFO,1
1,PASS,30
2,WARN,1


In [89]:
q('SELECT * FROM etl_quality_checks')

,check_id,layer,description,observed,expected,status
0,Q01,RAW,Volume : 1 si raw_container_events est vide (attendu 0),0.0,0.0,PASS
1,Q02,CORE,Unicite de event_id dans fact_container_event,0.0,0.0,PASS
2,Q03,CORE,Unicite de move_id dans fact_yard_move,0.0,0.0,PASS
3,Q04,STAGING,Valeurs nulles dans les colonnes cles de stg_container_events,0.0,0.0,PASS
4,Q05,CORE,Integrite referentielle : cles de fact_container_event absentes des dimensions,0.0,0.0,PASS
5,Q06,CORE,Integrite referentielle : cles de fact_yard_move absentes des dimensions,0.0,0.0,PASS
6,Q07,STAGING,Domaine des valeurs : event_type et freight_status hors referentiel,0.0,0.0,PASS
7,Q08,CORE,Plage du poids : valeurs hors 0..40000 kg dans fact_container_event,0.0,0.0,PASS
8,Q09,CORE,Reconciliation evenements : RAW - (CORE + rejets),0.0,0.0,PASS
9,Q10,CORE,Reconciliation mouvements : RAW - (CORE + rejets),0.0,0.0,PASS


**Interprétation.** Sur 32 contrôles : **30 PASS, 1 WARN, 1 INFO, 0 FAIL**.
- **WARN (Q13) :** 712 mouvements (15,8 %) ont la même zone au départ et à l'arrivée. Ils sont conservés et signalés : les supprimer reposerait sur un doute.
- **INFO (Q15) :** 1 481 événements Y08B ont été rattachés à Y08.

### 5.5 Réconciliation
Pour chaque source : `lignes RAW = lignes chargées dans le CORE + lignes rejetées` (contrôles Q09 et Q10).

In [90]:
# RAW = chargé dans le CORE + rejeté
q('''
SELECT 'container_events' AS source,
       (SELECT COUNT(*) FROM raw_container_events) AS raw,
       (SELECT COUNT(*) FROM fact_container_event) AS charge,
       (SELECT COUNT(DISTINCT raw_rowid) FROM etl_rejects WHERE source_table='container_events') AS rejete
UNION ALL
SELECT 'yard_moves',
       (SELECT COUNT(*) FROM raw_yard_moves),
       (SELECT COUNT(*) FROM fact_yard_move),
       (SELECT COUNT(DISTINCT raw_rowid) FROM etl_rejects WHERE source_table='yard_moves')
''')

,source,raw,charge,rejete
0,container_events,9000,9000,0
1,yard_moves,4500,4500,0


**Interprétation.** L'écart est nul pour les deux faits : aucune ligne n'a disparu sans trace.

## 6. Monitoring
Le journal `etl_run_metrics` enregistre, pour chaque étape : volume, durée et statut. Il est **conservé** d'une exécution à l'autre.

In [91]:
q(f"SELECT layer, step, rows_affected, ROUND(duration_s*1000, 1) AS duree_ms, status FROM etl_run_metrics WHERE run_id = '{run_id}' AND layer <> 'RAW'")

,layer,step,rows_affected,duree_ms,status
0,STAGING,stg_customers,80.0,3.1,OK
1,STAGING,stg_vessels,120.0,1.9,OK
2,STAGING,stg_equipment,40.0,1.6,OK
3,STAGING,create etl_rejects,NaN,2.0,OK
4,STAGING,EV01 event_id manquant,0.0,2.5,OK
5,STAGING,EV02 event_id en double,0.0,7.6,OK
6,STAGING,EV03 date invalide,0.0,1.5,OK
7,STAGING,EV04 type hors domaine,0.0,1.9,OK
8,STAGING,EV05 statut FCL/MTY invalide,0.0,1.2,OK
9,STAGING,EV06 poids hors plage,0.0,1.5,OK


Durée et volume par couche :

In [92]:
q(f"SELECT layer, COUNT(*) AS nb_etapes, ROUND(SUM(duration_s), 3) AS duree_s FROM etl_run_metrics WHERE run_id = '{run_id}' GROUP BY layer")

,layer,nb_etapes,duree_s
0,CORE,20,0.072
1,MART,7,0.047
2,RAW,5,0.304
3,STAGING,21,0.064


**Interprétation.** Le journal compte 53 étapes (5 chargements RAW et 48 étapes SQL). La durée varie d'une machine à l'autre ; sur ce volume elle reste de l'ordre d'une seconde ou moins. Une hausse brutale de la durée ou une baisse du nombre de lignes signalerait un problème.

## 7. Idempotence
**Idempotent** veut dire : relancer le pipeline donne le même résultat, sans doublons.

### 7.1 Rejouer tout le pipeline
Pendant ce notebook, `STEPS` et `CHECKS` ont mémorisé toutes les étapes. La fonction `rejouer_pipeline` les **rejoue** d'un coup sur une base (sans rien afficher).

In [93]:
# Rejoue les étapes mémorisées sur la base indiquée
def rejouer_pipeline(data_dir=DATA_DIR, db_path=DB_PATH):
    c = connect(db_path)
    reset_layers(c)
    rid = new_run_id()
    load_raw(c, rid, data_dir, verbose=False)
    for layer, name, sql in STEPS:
        run_step(c, rid, layer, name, sql, register=False, verbose=False)
    init_quality_table(c)
    statuts = [controle(c, *chk, register=False, verbose=False) for chk in CHECKS]
    c.close()
    return statuts

print(len(STEPS), "étapes SQL et", len(CHECKS), "contrôles mémorisés")

48 étapes SQL et 32 contrôles mémorisés


### 7.2 Test : deux exécutions, même résultat ?

In [94]:
def etat():
    return {t: con.execute(f"SELECT COUNT(*) FROM {t}").fetchone()[0]
            for t in ["fact_container_event", "fact_yard_move", "dim_customer", "etl_rejects"]}

avant = etat()
runs_avant = con.execute("SELECT COUNT(DISTINCT run_id) FROM etl_run_metrics").fetchone()[0]
time.sleep(1.1)                                   # pour obtenir un run_id différent
rejouer_pipeline()
apres = etat()
runs_apres = con.execute("SELECT COUNT(DISTINCT run_id) FROM etl_run_metrics").fetchone()[0]

print("Avant :", avant)
print("Après :", apres)
print("Résultat identique :", avant == apres)
print(f"Exécutions dans le journal : {runs_avant} -> {runs_apres}")

Avant : {'fact_container_event': 9000, 'fact_yard_move': 4500, 'dim_customer': 80, 'etl_rejects': 0}
Après : {'fact_container_event': 9000, 'fact_yard_move': 4500, 'dim_customer': 80, 'etl_rejects': 0}
Résultat identique : True
Exécutions dans le journal : 1 -> 2


**Interprétation.** Les volumes sont identiques avant et après : relancer le pipeline ne crée aucun doublon. Seul le journal s'allonge.

## 8. Preuve que les règles de rejet fonctionnent
Les données du prof sont propres, donc `etl_rejects` est vide. Pour **prouver** que les règles marchent, on fabrique une copie des CSV à laquelle on ajoute **13 lignes volontairement défectueuses**, puis on rejoue le pipeline dessus (base séparée `smartport_test.db`).

In [95]:
import tempfile
WORK = Path(tempfile.mkdtemp(prefix="smartport_test_"))   # dossier jetable : les sources ne sont pas modifiées
BAD, TEST_DB = WORK / "bad", WORK / "smartport_test.db"
BAD.mkdir()
for s in SOURCES:
    shutil.copy(DATA_DIR / f"{s}.csv", BAD / f"{s}.csv")

e = pd.read_csv(BAD / "container_events.csv")
m = pd.read_csv(BAD / "yard_moves.csv")
b = e.iloc[0].to_dict()
def ev(**kw): d = dict(b); d.update(kw); return d

mauvais_events = [
    ev(event_id=e.event_id.iloc[1]),                 # EV02 : identifiant en double
    ev(event_id=99001, event_ts="pas-une-date"),     # EV03 : date illisible
    ev(event_id=99002, event_type="TELEPORT"),       # EV04 : type inconnu
    ev(event_id=99003, freight_status="XXX"),        # EV05 : statut invalide
    ev(event_id=99004, gross_weight_kg=-50),         # EV06 : poids négatif
    ev(event_id=99005, gross_weight_kg=99999),       # EV06 : poids trop grand
    ev(event_id=99006, customer_id="C_INCONNU"),     # EV07 : client inexistant
    ev(event_id=99007, vessel_id="V_INCONNU"),       # EV08 : navire inexistant
    ev(event_id=99008, yard_area=""),                # EV09 : zone vide
]
mb = m.iloc[0].to_dict()
mauvais_moves = [
    dict(mb, move_id=m.move_id.iloc[1]),                    # MV02 : identifiant en double
    dict(mb, move_id=88001, move_ts="31/31/2026"),          # MV03 : date illisible
    dict(mb, move_id=88002, equipment_id="EQ_INCONNU"),     # MV04 : équipement inexistant
    dict(mb, move_id=88003, container_id="CONT_INCONNU"),   # MV06 : conteneur inexistant
]
pd.concat([e, pd.DataFrame(mauvais_events)]).to_csv(BAD / "container_events.csv", index=False)
pd.concat([m, pd.DataFrame(mauvais_moves)]).to_csv(BAD / "yard_moves.csv", index=False)

statuts = rejouer_pipeline(BAD, TEST_DB)
print("Contrôles en échec (FAIL) sur les données défectueuses :", statuts.count("FAIL"), "(attendu : ils détectent les erreurs injectées)")

Contrôles en échec (FAIL) sur les données défectueuses : 4 (attendu : ils détectent les erreurs injectées)


In [96]:
tc = sqlite3.connect(TEST_DB)
print("Lignes rejetées, par règle :")
display(pd.read_sql("SELECT rule_id, rule_description, COUNT(*) AS nb FROM etl_rejects GROUP BY 1, 2 ORDER BY 1", tc))
print("Faits chargés malgré les 13 erreurs injectées :")
display(pd.read_sql("""SELECT (SELECT COUNT(*) FROM fact_container_event) AS evenements,
                       (SELECT COUNT(*) FROM fact_yard_move) AS mouvements""", tc))
print("Contrôles qui détectent les erreurs injectées (donnée source) :")
display(pd.read_sql("SELECT check_id, layer, status, observed FROM etl_quality_checks WHERE status <> 'PASS' ORDER BY layer, check_id", tc))
print("Réconciliation (écart attendu 0) :")
display(pd.read_sql("SELECT check_id, status, observed FROM etl_quality_checks WHERE check_id IN ('Q09','Q10')", tc))

Lignes rejetées, par règle :


,rule_id,rule_description,nb
0,EV02,event_id en double (on garde la 1re ligne),1
1,EV03,event_ts illisible,1
2,EV04,event_type hors domaine,1
3,EV05,freight_status hors FCL/MTY,1
4,EV06,poids non numerique ou hors 0..40000 kg,2
5,EV07,customer_id absent du referentiel,1
6,EV08,vessel_id absent du referentiel,1
7,EV09,yard_area manquante,1
8,MV02,move_id en double (on garde la 1re ligne),1
9,MV03,move_ts illisible,1


Faits chargés malgré les 13 erreurs injectées :


,evenements,mouvements
0,9000,4500


Contrôles qui détectent les erreurs injectées (donnée source) :


,check_id,layer,status,observed
0,Q13,CORE,WARN,712.0
1,DQ_DUPLICATE_CHECK_container_events,RAW,FAIL,1.0
2,DQ_DUPLICATE_CHECK_yard_moves,RAW,FAIL,1.0
3,FK_CONTAINER,RAW,FAIL,1.0
4,FK_EQUIPMENT,RAW,FAIL,1.0
5,Q15,STAGING,INFO,1481.0


Réconciliation (écart attendu 0) :


,check_id,status,observed
0,Q09,PASS,0.0
1,Q10,PASS,0.0


**À retenir :** les 13 lignes défectueuses sont toutes rejetées par la bonne règle, les faits restent à 9 000 événements et 4 500 mouvements, et la réconciliation reste à 0. Les contrôles placés sur la donnée source (doublons, clés orphelines) passent en `FAIL` : ils **détectent** les erreurs. Les contrôles placés sur le CORE restent à `PASS` : les lignes défectueuses ont été **rejetées** et n'ont pas pollué le DWH. Le pipeline est donc protégé contre des données sales, même si les données fournies ne le sont pas.

## 9. Lineage : d'où vient chaque donnée ?
Le **lineage** répond à deux questions : *d'où vient ce chiffre ?* et *qu'est-ce qui sera touché si une source change ?*

```
CSV ──► raw_*  ──► stg_*  ──► dim_* / fact_*  ──► mart_kpi_*  ──► EDA (notebook 02)
 │                   │                │
 │                   └─► etl_rejects  └──► plateforme Streamlit (calcule ses KPI sur le CORE)
 └──► HDFS : archive de container_events.csv (chargement via HDFS : 04_scripts/hdfs/ingestion_depuis_hdfs.py)
```
### 9.1 Lineage au niveau colonne, vérifié sur le schéma réel
`04_scripts/lineage/lineage.py` décrit chaque colonne et compare cette description à la base : colonne oubliée, colonne disparue ou origine inexistante provoquent une erreur. Il ne vérifie pas la formule elle-même.

In [97]:
import sys
sys.path.insert(0, str(ROOT / "04_scripts" / "lineage"))
import lineage
ecarts = lineage.verifier(DB_PATH)
assert not ecarts, ecarts
LIN = pd.DataFrame(lineage.lignes())
LIN.to_csv(ROOT / "05_governance" / "lineage_colonnes.csv", index=False)   # export pour la gouvernance
print(f"{LIN.table_cible.nunique()} tables et {len(LIN)} colonnes décrites, 0 écart avec le schéma")
LIN[LIN.table_cible == "mart_kpi_move_by_area"][["colonne_cible", "origines_directes", "transformation", "origines_finales"]]

27 tables et 162 colonnes décrites, 0 écart avec le schéma


,colonne_cible,origines_directes,transformation,origines_finales
127,area_code,dim_yard_area.area_code,Axe d'agrégation,container_events.csv.yard_area ; yard_moves.csv.from_area ; yard_moves.csv.to_area
128,parent_area,dim_yard_area.parent_area,Axe d'agrégation,container_events.csv.yard_area ; yard_moves.csv.from_area ; yard_moves.csv.to_area
129,is_subarea,dim_yard_area.is_subarea,Axe d'agrégation,container_events.csv.yard_area ; yard_moves.csv.from_area ; yard_moves.csv.to_area
130,nb_move,fact_container_event.event_id ; dim_event_type.event_type,COUNT(*) où event_type = MOVE,container_events.csv.event_id ; container_events.csv.event_type


### 9.2 Analyse d'impact
Quels objets dépendent d'une colonne source ? Exemple : `yard_area` dans `container_events.csv`.

In [98]:
cible = "container_events.csv.yard_area"
impact = LIN[LIN.origines_finales.str.split(" ; ").apply(lambda l: cible in l) & (LIN.couche != "RAW")]
impact.groupby(["couche", "table_cible"], sort=False).colonne_cible.apply(", ".join).reset_index()

,couche,table_cible,colonne_cible
0,STAGING,stg_container_events,"yard_area_raw, yard_area_std"
1,CORE,dim_yard_area,"area_code, parent_area, is_subarea"
2,CORE,fact_container_event,area_key
3,CORE,fact_yard_move,"from_area_key, to_area_key"
4,MART,mart_kpi_move_by_area,"area_code, parent_area, is_subarea"


**Interprétation.** Un changement de codification des zones touche la zone standardisée, la dimension des zones, la clé de zone du fait et `mart_kpi_move_by_area`. Il faut vérifier ces objets avant de modifier la source.

### 9.3 Traçabilité d'une ligne
On suit un événement depuis le CSV jusqu'à l'indicateur auquel il contribue.

In [99]:
EID = 1
csv_ligne = pd.read_csv(DATA_DIR / "container_events.csv", dtype=str).query("event_id == @EID.__str__()")
print("1. CSV"); display(csv_ligne)
print("2. RAW");     display(q(f"SELECT rowid AS raw_rowid, * FROM raw_container_events WHERE event_id = '{EID}'"))
print("3. STAGING"); display(q(f"SELECT * FROM stg_container_events WHERE event_id = {EID}"))
print("4. CORE");    display(q(f"SELECT * FROM fact_container_event WHERE event_id = {EID}"))
print("5. MART : la ligne du jour auquel l'événement contribue")
display(q(f"""SELECT m.* FROM mart_kpi_events_per_day m JOIN dim_date d ON d.full_date = m.full_date
             JOIN fact_container_event f ON f.date_key = d.date_key WHERE f.event_id = {EID}"""))

1. CSV


,event_id,container_id,customer_id,vessel_id,event_type,event_ts,yard_area,freight_status,gross_weight_kg
0,1,CONT0003316,C050,V030,MOVE,2026-03-25 01:47:32,Y08B,FCL,8822.5


2. RAW


,raw_rowid,event_id,container_id,customer_id,vessel_id,event_type,event_ts,yard_area,freight_status,gross_weight_kg,_source_file,_loaded_at
0,1,1,CONT0003316,C050,V030,MOVE,2026-03-25 01:47:32,Y08B,FCL,8822.5,container_events.csv,2026-11-25T21:46:50


3. STAGING


,raw_rowid,event_id,container_id,customer_id,vessel_id,event_type,event_ts,yard_area_raw,yard_area_std,freight_status,gross_weight_kg
0,1,1,CONT0003316,C050,V030,MOVE,2026-03-25 01:47:32,Y08B,Y08,FCL,8822.5


4. CORE


,event_id,container_id,date_key,hour_key,customer_key,vessel_key,area_key,event_type_key,freight_key,event_ts,event_count,gross_weight_kg
0,1,CONT0003316,20260325,1,50,30,6,4,1,2026-03-25 01:47:32,1,8822.5


5. MART : la ligne du jour auquel l'événement contribue


,full_date,year_month,nb_evenements,nb_conteneurs,tonnes
0,2026-03-25,2026-03,38,38,650.4


### 9.4 Preuve par les volumes
La somme des événements de chaque table du MART doit retrouver le volume du fait (les MOVE seuls pour la table des zones).

In [100]:
attendu = {t: "SELECT COUNT(*) FROM fact_container_event" for t in
           ["mart_kpi_events_per_day", "mart_kpi_vessel", "mart_kpi_service", "mart_kpi_customer", "mart_kpi_event_type", "mart_kpi_freight"]}
attendu["mart_kpi_move_by_area"] = ("SELECT COUNT(*) FROM fact_container_event f JOIN dim_event_type t "
                                    "ON t.event_type_key = f.event_type_key WHERE t.event_type = 'MOVE'")
res = []
for t, sql in attendu.items():
    col = "nb_move" if t.endswith("by_area") else "nb_evenements"
    obtenu = con.execute(f"SELECT SUM({col}) FROM {t}").fetchone()[0]
    res.append((t, obtenu, con.execute(sql).fetchone()[0]))
res = pd.DataFrame(res, columns=["table_mart", "somme_mart", "attendu_core"])
assert (res.somme_mart == res.attendu_core).all()
res

,table_mart,somme_mart,attendu_core
0,mart_kpi_events_per_day,9000,9000
1,mart_kpi_vessel,9000,9000
2,mart_kpi_service,9000,9000
3,mart_kpi_customer,9000,9000
4,mart_kpi_event_type,9000,9000
5,mart_kpi_freight,9000,9000
6,mart_kpi_move_by_area,2840,2840


**Interprétation.** Les sept tables du MART sont cohérentes avec le CORE.
**Limite :** une ligne acceptée se retrouve par sa clé métier (`event_id`, `move_id`) ; les faits ne portent ni le numéro de ligne RAW ni l'identifiant d'exécution (`run_id` n'existe que dans le journal).

## 10. Conclusion
| Question | Réponse mesurée |
|---|---|
| Les données sont-elles complètes ? | Oui : RAW = CORE + rejets, écart 0 |
| Sont-elles propres ? | 0 rejet ; 712 mouvements à instruire (WARN) |
| Les règles protègent-elles l'entrepôt ? | Oui : 13 erreurs injectées sur 13 rejetées |
| Peut-on relancer le pipeline ? | Oui : mêmes volumes, aucun doublon |
| Peut-on expliquer un chiffre ? | Oui : lineage au niveau colonne et trace d'une ligne |

**Limites :** reconstruction complète à chaque exécution (pas d'ingestion incrémentale) ; données synthétiques ; SQLite sur un seul poste ; le notebook lit les CSV locaux (le chargement via HDFS passe par `04_scripts/hdfs/ingestion_depuis_hdfs.py`).

**Suite :** exécuter `02_EDA_Analyse_Exploratoire.ipynb`.

In [101]:
con.close()